# Ejercicio 2: Implementación de la Distancia de Gower con Análisis Sociodemográfico

El script debe contener una implementación de la Distancia de Gower enfocada en características sociodemográficas:

1. Implementar la fórmula de Gower.
2. Validar con un subconjunto pequeño (ej. 5 filas).
3. Probar el cálculo de la matriz de distancias en una muestra pequeña del dataset.
4. **ANÁLISIS SOCIODEMOGRÁFICO**: Analizar las características sociodemográficas (income, age, sex) de las muestras para identificar patrones de ingresos.

In [1]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
from IPython.display import display

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# =================================================================
# Carga del DataFrame desde archivos locales
# =================================================================
df = cargar_dataset_adult()

# =================================================================
# Implementación Vectorizada de la Distancia de Gower
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Tomamos una muestra de 5 filas específicas
sample_indices = [29, 30, 31, 32, 33]
df_sample = df.loc[sample_indices].copy()

print(f"--- Se ha tomado una muestra de {len(df_sample)} filas (índices: {sample_indices}) ---")

# Identificar tipos de variables
# Nota: fnlwgt (final weight) NO se incluye porque es un peso de muestreo del censo,
# no una característica descriptiva. No tiene sentido en el cálculo de similitud.
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# --- Manejo de la Variable Ordinal 'education' ---
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Cálculo Vectorizado de la Matriz de Distancias
# -----------------------------------------------------------------
print("\nCalculando la matriz de distancias (VERSIÓN VECTORIZADA)...\n")

# Paso A: Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Paso B: Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Paso C: Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Paso D: Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Paso E: Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix_vectorized = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)


# 3. Visualización de Resultados
# -----------------------------------------------------------------
gower_matrix_df = pd.DataFrame(gower_matrix_vectorized, 
                               index=df_sample.index, 
                               columns=df_sample.index)

print("--- Validación de la Distancia de Gower con Variables Ordinales ---")
print("Matriz de Distancias de Gower resultante:\n")
display(gower_matrix_df)

--- Se ha tomado una muestra de 5 filas (índices: [29, 30, 31, 32, 33]) ---

Calculando la matriz de distancias (VERSIÓN VECTORIZADA)...

--- Validación de la Distancia de Gower con Variables Ordinales ---
Matriz de Distancias de Gower resultante:



,29,30,31,32,33
29,0.0000,0.3252,0.3048,0.2608,0.2262
30,0.3252,0.0000,0.2926,0.3254,0.2990
31,0.3048,0.2926,0.0000,0.2738,0.2785
32,0.2608,0.3254,0.2738,0.0000,0.2619
33,0.2262,0.2990,0.2785,0.2619,0.0000


Calculo con una muestra pequeña:

In [2]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
import time # Para medir el tiempo de ejecución
import psutil # Para medir el uso de memoria
import os # Para obtener el ID del proceso actual

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# =================================================================
# Carga del DataFrame desde archivos locales
# =================================================================
df = cargar_dataset_adult()

# =================================================================
# Implementación Vectorizada de Gower para una MUESTRA
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Define el tamaño de la muestra
sample_size = 400

# Tomamos una muestra aleatoria del DataFrame
df_sample = df.sample(n=sample_size, random_state=42).copy()

print(f"--- Se ha tomado una muestra aleatoria de {sample_size} filas. ---")

# Identificar tipos de variables de forma definitiva
# Nota: fnlwgt (final weight) NO se incluye porque es un peso de muestreo del censo,
# no una característica descriptiva. No tiene sentido en el cálculo de similitud.
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# --- Manejo de la Variable Ordinal 'education' ---
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Cálculo Vectorizado de la Matriz de Distancias
# -----------------------------------------------------------------
print("\nCalculando la matriz de distancias (VERSIÓN VECTORIZADA)...")

# --- INICIO DE MEDICIÓN DE RECURSOS ---
process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 * 1024) # Memoria en MB
start_time = time.time()
print(f"Uso de memoria antes del cálculo: {mem_before:.2f} MB")
# ------------------------------------


# Paso A: Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Paso B: Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Paso C: Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Paso D: Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Paso E: Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix_vectorized = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)


# --- FIN DE MEDICIÓN DE RECURSOS ---
end_time = time.time()
mem_after = process.memory_info().rss / (1024 * 1024) # Memoria en MB

print("\n--- Resumen de Ejecución (Vectorizado) ---")
print(f"¡Cálculo completado en {end_time - start_time:.4f} segundos!")
print(f"Uso de memoria final: {mem_after:.2f} MB")
print(f"Memoria adicional utilizada por el cálculo: {mem_after - mem_before:.2f} MB")
# ---------------------------------


# 3. Visualización de Resultados
# -----------------------------------------------------------------
gower_matrix_df = pd.DataFrame(gower_matrix_vectorized, index=df_sample.index, columns=df_sample.index)
print("\n--- Matriz de Distancias de Gower resultante ---")
print("(Mostrando las primeras 10x10 para visualización)")

# Retornar el DataFrame lo mostrará en formato tabla
gower_matrix_df.iloc[:10, :10]

# ============================================================================
# ANÁLISIS SOCIODEMOGRÁFICO DE LA MUESTRA
# ============================================================================

print("\n" + "="*70)
print("ANÁLISIS SOCIODEMOGRÁFICO DE LA MUESTRA")
print("="*70)

print(f"\nMuestra analizada: {len(df_sample)} observaciones")
print("Variables sociodemográficas clave: income, age, sex")

# 1. ANÁLISIS DE INGRESOS
print("\n" + "="*50)
print("1. ANÁLISIS DE INGRESOS")
print("="*50)

income_dist = df_sample['income'].value_counts()
print("\nDistribución de ingresos en la muestra:")
print(income_dist)

if len(income_dist) > 1:
    high_income_pct = (income_dist.get('>50K', 0) / len(df_sample)) * 100
    print(f"\nPorcentaje de ingresos altos (>$50K): {high_income_pct:.1f}%")

# 2. ANÁLISIS DE EDAD
print("\n" + "="*50)
print("2. ANÁLISIS DE EDAD")
print("="*50)

age_stats = df_sample['age'].describe()
print("\nEstadísticas de edad en la muestra:")
print(f"Edad promedio: {age_stats['mean']:.1f} años")
print(f"Edad mínima: {age_stats['min']:.0f} años")
print(f"Edad máxima: {age_stats['max']:.0f} años")
print(f"Desviación estándar: {age_stats['std']:.1f} años")

# 3. ANÁLISIS DE GÉNERO
print("\n" + "="*50)
print("3. ANÁLISIS DE GÉNERO")
print("="*50)

gender_dist = df_sample['sex'].value_counts()
print("\nDistribución de género en la muestra:")
print(gender_dist)

if len(gender_dist) > 1:
    male_pct = (gender_dist.get('Male', 0) / len(df_sample)) * 100
    female_pct = (gender_dist.get('Female', 0) / len(df_sample)) * 100
    print(f"\nPorcentajes:")
    print(f"Masculino: {male_pct:.1f}%")
    print(f"Femenino: {female_pct:.1f}%")

# 4. ANÁLISIS CONJUNTO SOCIODEMOGRÁFICO
print("\n" + "="*50)
print("4. ANÁLISIS CONJUNTO SOCIODEMOGRÁFICO")
print("="*50)

print("\nDistribución de ingresos por género:")
gender_income = pd.crosstab(df_sample['sex'], df_sample['income'], margins=True)
print(gender_income)

# Análisis de edad por ingresos
print("\nEstadísticas de edad por nivel de ingresos:")
age_by_income = df_sample.groupby('income')['age'].agg(['mean', 'count'])
print(age_by_income.round(1))

# 5. PERFILES SOCIODEMOGRÁFICOS
print("\n" + "="*50)
print("5. PERFILES SOCIODEMOGRÁFICOS DE LA MUESTRA")
print("="*50)

print("\nResumen sociodemográfico:")
print(f"• Tamaño de muestra: {len(df_sample)} observaciones")
if len(income_dist) > 1:
    print(f"• Ingresos altos: {high_income_pct:.1f}%")
print(f"• Edad promedio: {age_stats['mean']:.1f} años")
if len(gender_dist) > 1:
    print(f"• Género masculino: {male_pct:.1f}%")

# 6. RELACIÓN CON DISTANCIAS DE GOWER
print("\n" + "="*50)
print("6. RELACIÓN CON DISTANCIAS DE GOWER")
print("="*50)

print("\nLa matriz de distancias de Gower calculada incluye:")
print("• Variables numéricas: age, capital-gain, capital-loss, hours-per-week, education-num")
print("• Variables categóricas: workclass, marital-status, occupation, relationship, race, sex, native-country, income")
print("• Variables ordinales: education")

print("\nLas distancias más bajas indican mayor similitud sociodemográfica entre observaciones.")

print(f"\n" + "="*70)
print("FIN DEL ANÁLISIS SOCIODEMOGRÁFICO")
print("="*70)

--- Se ha tomado una muestra aleatoria de 400 filas. ---

Calculando la matriz de distancias (VERSIÓN VECTORIZADA)...
Uso de memoria antes del cálculo: 135.00 MB

--- Resumen de Ejecución (Vectorizado) ---
¡Cálculo completado en 0.0222 segundos!
Uso de memoria final: 137.86 MB
Memoria adicional utilizada por el cálculo: 2.87 MB

--- Matriz de Distancias de Gower resultante ---
(Mostrando las primeras 10x10 para visualización)

ANÁLISIS SOCIODEMOGRÁFICO DE LA MUESTRA

Muestra analizada: 400 observaciones
Variables sociodemográficas clave: income, age, sex

1. ANÁLISIS DE INGRESOS

Distribución de ingresos en la muestra:
income
<=50K    314
>50K      86
Name: count, dtype: int64

Porcentaje de ingresos altos (>$50K): 21.5%

2. ANÁLISIS DE EDAD

Estadísticas de edad en la muestra:
Edad promedio: 39.8 años
Edad mínima: 17 años
Edad máxima: 90 años
Desviación estándar: 14.1 años

3. ANÁLISIS DE GÉNERO

Distribución de género en la muestra:
sex
Male      256
Female    144
Name: count, dtype: